# Classical pruning baseline (greedy sensitivity ranking)

This is the classical counterpart to the QUBO/QAOA pruning decision, so the
two can be compared head-to-head.

**Same candidate pool, same target percentage, different decision method:**
it reads the exact same `qubo_outputs/selected_candidates.csv` (the 10
candidates) and the exact same `target_compression` from
`qubo_outputs/qubo_metadata.json` that the QUBO/QAOA run used. Instead of
building a Hamiltonian and optimizing/sampling a quantum circuit, it just
**sorts candidates by pruning_attractiveness and greedily accumulates them
until the target compression is reached** — this is how structured pruning
is normally done in practice, with no optimizer at all.

It then applies the resulting mask to the same real fine-tuned model and
evaluates on the same real data, using the same evaluation code path as
`top_k_mask_evaluation.ipynb`, so accuracy/loss/F1 numbers are directly
comparable. Both the **selection time** (this notebook) and the QAOA
**selection time** (`qaoa_ranked_masks.json`) are saved so
`quantum_vs_classical_comparison.ipynb` can compare decision-making cost,
not just outcome quality.

## Step 1 — Imports and configuration

In [1]:
from __future__ import annotations

import csv
import gc
import json
import time
from pathlib import Path
from typing import Any, Dict, List, Tuple

import numpy as np
import torch
import torch.nn as nn
from PIL import Image
from sklearn.metrics import accuracy_score, f1_score
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from torchvision.transforms import v2
from tqdm import tqdm

import timm
from datasets import load_dataset
from huggingface_hub import hf_hub_download

PROJECT_DIR = Path.cwd()
OUTPUT_DIR = PROJECT_DIR / "qubo_outputs"

CLASS_NAMES = [
    "Calgary", "Charlottetown", "Edmonton", "Halifax", "Hamilton",
    "Kitchener-Waterloo", "Montreal", "Ottawa-Gatineau", "Quebec City",
    "Saskatoon", "St Johns", "Toronto", "Vancouver", "Victoria", "Winnipeg",
]

MODEL = "convnext"
SPLIT = "test"
MAX_SAMPLES = 600
BATCH_SIZE = 16
NUM_WORKERS = 0

SELECTED_CANDIDATES_CSV = OUTPUT_DIR / "selected_candidates.csv"
QUBO_METADATA_JSON = OUTPUT_DIR / "qubo_metadata.json"
QUBO_MATRIX_CSV = OUTPUT_DIR / "qubo_matrix.csv"
QUBO_TERMS_JSON = OUTPUT_DIR / "qubo_terms.json"

OUTPUT_CSV = OUTPUT_DIR / "classical_pruning_result.csv"
OUTPUT_JSON = OUTPUT_DIR / "classical_pruning_result.json"

with QUBO_METADATA_JSON.open("r", encoding="utf-8") as f:
    qubo_metadata = json.load(f)

TARGET_COMPRESSION = qubo_metadata["target_compression"]
TARGET_TOLERANCE = qubo_metadata["target_tolerance"]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Project directory:", PROJECT_DIR)
print("Target compression:", TARGET_COMPRESSION, "+/-", TARGET_TOLERANCE)
print("Device:", device)

Project directory: C:\_projects\quantum_pruning
Target compression: 0.4 +/- 0.06
Device: cpu


## Step 2 — Basic helpers

In [2]:
def safe_float(value: Any, default: float = 0.0) -> float:
    try:
        if value is None:
            return default
        text = str(value).strip()
        if text == "":
            return default
        return float(text)
    except Exception:
        return default


def safe_int(value: Any, default: int = 0) -> int:
    try:
        if value is None:
            return default
        text = str(value).strip()
        if text == "":
            return default
        return int(float(text))
    except Exception:
        return default


def read_csv(path: Path) -> List[Dict[str, Any]]:
    if not path.exists():
        raise FileNotFoundError(f"Missing CSV file: {path}")
    with path.open("r", newline="", encoding="utf-8") as file:
        return list(csv.DictReader(file))


def write_csv(path: Path, rows: List[Dict[str, Any]], fieldnames: List[str]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", newline="", encoding="utf-8") as file:
        writer = csv.DictWriter(file, fieldnames=fieldnames)
        writer.writeheader()
        for row in rows:
            writer.writerow(row)


def write_json(path: Path, data: Dict[str, Any]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as file:
        json.dump(data, file, indent=2)

## Step 3 — Greedy sensitivity-ranking selection (the "classical" decision)

No Hamiltonian, no optimizer, no sampling. Sort the same candidates by
`pruning_attractiveness` (high compression value per unit of loss risk,
computed cleanly from the already-normalized `compression_value`/
`loss_penalty` columns) and keep adding the next-most-attractive block
until cumulative compression reaches the target. This mirrors real-world
structured pruning practice.

In [3]:
candidates = read_csv(SELECTED_CANDIDATES_CSV)

for row in candidates:
    row["qubit_index"] = safe_int(row["qubit_index"])
    row["loss_penalty"] = safe_float(row["loss_penalty"])
    row["compression_value"] = safe_float(row["compression_value"])
    row["params"] = safe_int(row["params"])
    row["pruning_attractiveness"] = row["compression_value"] / (row["loss_penalty"] + 1e-6)

candidates_by_qubit = sorted(candidates, key=lambda r: r["qubit_index"])
n = len(candidates_by_qubit)

selection_start = time.perf_counter()

ranked_for_pruning = sorted(candidates, key=lambda r: r["pruning_attractiveness"], reverse=True)

pruned_qubit_indices = set()
cumulative_compression = 0.0

for row in ranked_for_pruning:
    if cumulative_compression >= TARGET_COMPRESSION:
        break
    pruned_qubit_indices.add(row["qubit_index"])
    cumulative_compression += row["compression_value"]

selection_seconds = time.perf_counter() - selection_start

bitstring = "".join("1" if c["qubit_index"] in pruned_qubit_indices else "0" for c in candidates_by_qubit)
pruned_blocks = [c["candidate"] for c in candidates_by_qubit if c["qubit_index"] in pruned_qubit_indices]
predicted_loss_penalty = sum(c["loss_penalty"] for c in candidates_by_qubit if c["qubit_index"] in pruned_qubit_indices)

print("Greedy ranking (most attractive to prune first):")
for row in ranked_for_pruning:
    print(f"  qubit={row['qubit_index']:>2} candidate={row['candidate']:<20} "
          f"attractiveness={row['pruning_attractiveness']:.4f} compression={row['compression_value']:.4f}")

print(f"\nSelection time: {selection_seconds:.6f}s")
print(f"Bitstring: {bitstring}")
print(f"Pruned blocks: {'; '.join(pruned_blocks)}")
print(f"Predicted compression: {cumulative_compression:.4f} (target {TARGET_COMPRESSION} +/- {TARGET_TOLERANCE})")
print(f"Predicted loss penalty: {predicted_loss_penalty:.6f}")

Greedy ranking (most attractive to prune first):
  qubit= 0 candidate=stages.2.blocks.2    attractiveness=16.8869 compression=0.0751
  qubit= 1 candidate=stages.3.blocks.2    attractiveness=10.7567 compression=0.2977
  qubit= 2 candidate=stages.2.blocks.0    attractiveness=8.4897 compression=0.0751
  qubit= 3 candidate=stages.2.blocks.5    attractiveness=7.4996 compression=0.0751
  qubit= 4 candidate=stages.2.blocks.3    attractiveness=6.4144 compression=0.0751
  qubit= 5 candidate=stages.2.blocks.7    attractiveness=4.6421 compression=0.0751
  qubit= 6 candidate=stages.3.blocks.0    attractiveness=2.7969 compression=0.2977
  qubit= 7 candidate=stages.1.blocks.1    attractiveness=2.3070 compression=0.0191
  qubit= 8 candidate=stages.0.blocks.2    attractiveness=0.5063 compression=0.0050
  qubit= 9 candidate=stages.0.blocks.0    attractiveness=0.0050 compression=0.0050

Selection time: 0.000343s
Bitstring: 1110000000
Pruned blocks: stages.2.blocks.2; stages.3.blocks.2; stages.2.blocks.0

## Step 4 — QUBO energy of the classical mask (informational only)

The greedy heuristic never looks at the QUBO objective. This just scores the
resulting mask against the *same* Hamiltonian QAOA optimized, purely so the
comparison notebook can report whether the classical pick happens to be
close to, or far from, the true QUBO optimum -- a quality check, not part of
the classical method itself.

In [4]:
Q_rows = read_csv(QUBO_MATRIX_CSV)
Q = np.array([[safe_float(row[f"q{j}"]) for j in range(n)] for row in Q_rows], dtype=float)

with QUBO_TERMS_JSON.open("r", encoding="utf-8") as f:
    qubo_constant = safe_float(json.load(f).get("constant", 0.0))

x = np.array([1.0 if bit == "1" else 0.0 for bit in bitstring], dtype=float)
qubo_energy = float(qubo_constant + x @ Q @ x)

print("QUBO energy of the classical greedy mask:", qubo_energy)

QUBO energy of the classical greedy mask: -0.2323461227907373


## Step 5 — Image preprocessing, dataset, model loading, evaluation

Identical to `top_k_mask_evaluation.ipynb` / `qnn_and_pruning.ipynb`, duplicated
here so this notebook is self-contained.

In [5]:
def resize_and_pad(img: Image.Image, target_size=(320, 320)) -> Image.Image:
    img = img.copy()
    img.thumbnail(target_size, Image.Resampling.LANCZOS)
    new_img = Image.new("RGB", target_size, (0, 0, 0))
    left = (target_size[0] - img.size[0]) // 2
    top = (target_size[1] - img.size[1]) // 2
    new_img.paste(img, (left, top))
    return new_img


def get_transform(model_name: str):
    if model_name == "convnext":
        return v2.Compose([
            v2.Lambda(lambda img: resize_and_pad(img)),
            v2.ToImage(),
            v2.ToDtype(torch.float32, scale=True),
            v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ])
    if model_name == "swinv2":
        return transforms.Compose([
            transforms.Resize((192, 192)),
            transforms.ToTensor(),
            transforms.Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5)),
        ])
    raise ValueError(f"Unknown model name: {model_name}")


class StreetViewSubset(Dataset):
    def __init__(self, hf_dataset, transform):
        self.data = list(hf_dataset)
        self.transform = transform

    def __len__(self) -> int:
        return len(self.data)

    def __getitem__(self, idx: int):
        row = self.data[idx]
        img = row["image"]
        if not isinstance(img, Image.Image):
            img = Image.open(img)
        img = img.convert("RGB")
        x = self.transform(img)
        y = int(row["label"])
        return x, y


def build_dataloader(model_name, split, max_samples, batch_size, num_workers) -> DataLoader:
    transform = get_transform(model_name)
    split_expr = f"{split}[:{max_samples}]" if max_samples > 0 else split
    ds = load_dataset("canada-guesser/Canadian-streetview-cities", split=split_expr)
    wrapped = StreetViewSubset(ds, transform)
    return DataLoader(wrapped, batch_size=batch_size, shuffle=False,
                       num_workers=num_workers, pin_memory=torch.cuda.is_available())


def torch_load_compatible(path: str, device: torch.device):
    try:
        return torch.load(path, map_location=device, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=device)


def load_finetuned_model(model_name: str, device: torch.device) -> nn.Module:
    if model_name == "convnext":
        path = hf_hub_download(
            repo_id="canada-guesser/canadian_streetview_cities_models",
            filename="cnn_model/convnext_tiny_set_3_final.bin",
        )
        model = timm.create_model("convnext_tiny", pretrained=False, num_classes=len(CLASS_NAMES))
        checkpoint = torch_load_compatible(path, device)
        state_dict = checkpoint["model_state_dict"] if isinstance(checkpoint, dict) and "model_state_dict" in checkpoint else checkpoint
        model.load_state_dict(state_dict)
    elif model_name == "swinv2":
        path = hf_hub_download(
            repo_id="canada-guesser/canadian_streetview_cities_models",
            filename="vit_model/swinv2_base_window12_192_0_finetuned_canadian_streetview.bin",
        )
        model = timm.create_model("swinv2_base_window12_192", pretrained=False, num_classes=len(CLASS_NAMES))
        model.load_state_dict(torch_load_compatible(path, device))
    else:
        raise ValueError(f"Unknown model: {model_name}")
    model.to(device)
    model.eval()
    return model


@torch.no_grad()
def evaluate(model: nn.Module, loader: DataLoader, device: torch.device) -> Dict[str, float]:
    model.eval()
    criterion = nn.CrossEntropyLoss(reduction="sum")
    total_loss = 0.0
    total = 0
    preds: List[int] = []
    labels: List[int] = []
    for x, y in tqdm(loader, desc="evaluate", leave=False):
        x = x.to(device)
        y = y.to(device)
        logits = model(x)
        if hasattr(logits, "logits"):
            logits = logits.logits
        loss = criterion(logits, y)
        total_loss += float(loss.item())
        total += int(y.numel())
        preds.extend(torch.argmax(logits, dim=1).detach().cpu().tolist())
        labels.extend(y.detach().cpu().tolist())
    return {
        "loss": total_loss / max(total, 1),
        "accuracy": float(accuracy_score(labels, preds)),
        "macro_f1": float(f1_score(labels, preds, average="macro", zero_division=0)),
        "n_samples": total,
    }


def count_trainable_params(module: nn.Module) -> int:
    return sum(p.numel() for p in module.parameters() if p.requires_grad)


class CandidateBypassWrapper(nn.Module):
    def __init__(self, module: nn.Module):
        super().__init__()
        self.module = module

    def forward(self, x, *args, **kwargs):
        return x


def get_parent_and_child(model: nn.Module, module_name: str) -> Tuple[nn.Module, str]:
    parts = module_name.split(".")
    parent = model
    for part in parts[:-1]:
        parent = parent[int(part)] if part.isdigit() else getattr(parent, part)
    return parent, parts[-1]


def get_module_by_name(model: nn.Module, module_name: str) -> nn.Module:
    module = model
    for part in module_name.split("."):
        module = module[int(part)] if part.isdigit() else getattr(module, part)
    return module


def replace_module(model: nn.Module, module_name: str, new_module: nn.Module) -> None:
    parent, child_key = get_parent_and_child(model, module_name)
    if child_key.isdigit():
        parent[int(child_key)] = new_module
    else:
        setattr(parent, child_key, new_module)


def apply_pruning_mask(model: nn.Module, blocks: List[str]) -> None:
    for block_name in blocks:
        block_name = block_name.strip()
        if not block_name:
            continue
        original_module = get_module_by_name(model, block_name)
        replace_module(model, block_name, CandidateBypassWrapper(original_module))

## Step 6 — Evaluate baseline and the classical greedy mask on real data

In [6]:
params_by_candidate = {c["candidate"]: c["params"] for c in candidates_by_qubit}

print("Loading validation/test subset...")
loader = build_dataloader(MODEL, SPLIT, MAX_SAMPLES, BATCH_SIZE, NUM_WORKERS)

print("Evaluating fresh baseline model...")
baseline_model = load_finetuned_model(MODEL, device)
total_trainable_params = count_trainable_params(baseline_model)

baseline_start = time.perf_counter()
baseline_metrics = evaluate(baseline_model, loader, device)
baseline_eval_seconds = time.perf_counter() - baseline_start

print(f"Baseline: loss={baseline_metrics['loss']:.6f}, acc={baseline_metrics['accuracy']:.4f}, "
      f"f1={baseline_metrics['macro_f1']:.4f}, eval_time={baseline_eval_seconds:.2f}s")

del baseline_model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("\nEvaluating classical greedy-pruned model...")
pruned_model = load_finetuned_model(MODEL, device)
apply_pruning_mask(pruned_model, pruned_blocks)

eval_start = time.perf_counter()
pruned_metrics = evaluate(pruned_model, loader, device)
eval_seconds = time.perf_counter() - eval_start

pruned_params = sum(params_by_candidate.get(b, 0) for b in pruned_blocks)
actual_parameter_reduction = pruned_params / total_trainable_params if total_trainable_params > 0 else 0.0

accuracy_drop = baseline_metrics["accuracy"] - pruned_metrics["accuracy"]
f1_drop = baseline_metrics["macro_f1"] - pruned_metrics["macro_f1"]
loss_increase = pruned_metrics["loss"] - baseline_metrics["loss"]

print(f"Pruned: loss={pruned_metrics['loss']:.6f}, acc={pruned_metrics['accuracy']:.4f}, "
      f"f1={pruned_metrics['macro_f1']:.4f}, eval_time={eval_seconds:.2f}s")
print(f"Drops: Δloss={loss_increase:.6f}, Δacc={accuracy_drop:.4f}, Δf1={f1_drop:.4f}")
print(f"Parameter reduction: {pruned_params:,} / {total_trainable_params:,} = {actual_parameter_reduction:.4%}")

del pruned_model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

Loading validation/test subset...


Resolving data files:   0%|          | 0/27 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/27 [00:00<?, ?it/s]

Evaluating fresh baseline model...


evaluate:   0%|          | 0/38 [00:00<?, ?it/s]

...

evaluate:  92%|█████████▏| 35/38 [01:25<00:06,  2.24s/it]

evaluate:  95%|█████████▍| 36/38 [01:27<00:04,  2.23s/it]

evaluate:  97%|█████████▋| 37/38 [01:29<00:02,  2.21s/it]

evaluate: 100%|██████████| 38/38 [01:30<00:00,  1.87s/it]

Pruned: loss=0.241919, acc=0.9450, f1=0.9394, eval_time=90.79s
Drops: Δloss=0.153282, Δacc=0.0467, Δf1=0.0517
Parameter reduction: 7,166,976 / 27,831,663 = 25.7512%


## Step 7 — Save results

In [7]:
result_row = {
    "method": "classical_greedy_sensitivity_ranking",
    "mask": bitstring,
    "pruned_blocks": "; ".join(pruned_blocks),
    "qubo_energy": qubo_energy,
    "predicted_compression": cumulative_compression,
    "predicted_loss_penalty": predicted_loss_penalty,
    "num_pruned_blocks": len(pruned_blocks),

    "baseline_accuracy": baseline_metrics["accuracy"],
    "baseline_macro_f1": baseline_metrics["macro_f1"],
    "baseline_validation_loss": baseline_metrics["loss"],

    "actual_accuracy": pruned_metrics["accuracy"],
    "actual_macro_f1": pruned_metrics["macro_f1"],
    "actual_validation_loss": pruned_metrics["loss"],

    "accuracy_drop": accuracy_drop,
    "f1_drop": f1_drop,
    "loss_increase": loss_increase,

    "pruned_parameter_count": pruned_params,
    "total_trainable_params": total_trainable_params,
    "actual_parameter_reduction": actual_parameter_reduction,

    "n_samples": pruned_metrics["n_samples"],
    "selection_time_seconds": selection_seconds,
    "eval_seconds": eval_seconds,
}

fieldnames = list(result_row.keys())
write_csv(OUTPUT_CSV, [result_row], fieldnames)

summary = {
    "method": "classical_greedy_sensitivity_ranking",
    "model": MODEL,
    "split": SPLIT,
    "max_samples": MAX_SAMPLES,
    "target_compression": TARGET_COMPRESSION,
    "target_tolerance": TARGET_TOLERANCE,
    "baseline": baseline_metrics,
    "baseline_eval_seconds": baseline_eval_seconds,
    "total_trainable_params": total_trainable_params,
    "result": result_row,
    "note": (
        "Blocks were ranked by pruning_attractiveness (compression_value / loss_penalty) "
        "computed from the same selected_candidates.csv the QUBO/QAOA run used, and greedily "
        "accumulated until the same target_compression was reached. No Hamiltonian, optimizer, "
        "or sampling was involved -- selection_time_seconds is a sort plus a cumulative sum."
    ),
}

write_json(OUTPUT_JSON, summary)

print("Saved:", OUTPUT_CSV)
print("Saved:", OUTPUT_JSON)
print(json.dumps(result_row, indent=2))

Saved: C:\_projects\quantum_pruning\qubo_outputs\classical_pruning_result.csv
Saved: C:\_projects\quantum_pruning\qubo_outputs\classical_pruning_result.json
{
  "method": "classical_greedy_sensitivity_ranking",
  "mask": "1110000000",
  "pruned_blocks": "stages.2.blocks.2; stages.3.blocks.2; stages.2.blocks.0",
  "qubo_energy": -0.2323461227907373,
  "predicted_compression": 0.4479216665066718,
  "predicted_loss_penalty": 0.040967938725294396,
  "num_pruned_blocks": 3,
  "baseline_accuracy": 0.9916666666666667,
  "baseline_macro_f1": 0.9910687234689917,
  "baseline_validation_loss": 0.08863729625940323,
  "actual_accuracy": 0.945,
  "actual_macro_f1": 0.9394025753898378,
  "actual_validation_loss": 0.24191885073979696,
  "accuracy_drop": 0.046666666666666745,
  "f1_drop": 0.05166614807915382,
  "loss_increase": 0.15328155448039374,
  "pruned_parameter_count": 7166976,
  "total_trainable_params": 27831663,
  "actual_parameter_reduction": 0.2575115974923956,
  "n_samples": 600,
  "select